# Model evaluation workflow designed for WildObs Image Management Platform

## Description
- This script can be used for benchmarking an ai species recognition model with a local dataset to get an independent assessment of Recall, Precision and F1 Score for a given location
- The purpose is to evaluate the most suitable or best performing model for a given location and inform an appropriate validation workflow to ensure accuracy requirements are met

## Setup instructions:
1) Prepare the testing dataset. Start by organising camera trap images into folders by species on your local computer. The quality of the testing dataset will determine the accuracy of the report generated. Here are a few tips:
- Use a representive number of images of each species e.g. at least 1000 if possible
- Be sure to select images that have not been used as part of a model training dataset as these will create a biased result
- If possible, select a subset of testing images from a larger pool aiming to get a wide range of images over space and time
- For the purpose of calculating Recall, include species that are relevant to your monitoring program
- For the the purpose of calculating Precision, also include images of other species that are commonly detected on the cameras at that location even if they are not relevant to your monitoring. Also include some blank images. Below is an example breakdown:
  - Feral Cat x 1000 (Target species in monitoring program)
  - Red Fox x 1000 (Target species in monitoring program)
  - European Rabbit x 1000 (Target species in monitoring program)
  - Kangaroo x 1000 (Non-target species but commonly detected at this location)
  - Emu x 1000 (Non-target species but commonly detected at this location)
  - Blank x 1000 (Not relevant to monitoring but helps with calculation of Precision)
2) Establish new Project/s in the WildObs WIMP for benchmarking purposes:
-  You will need a separate Project for each model you are testing. 
-  Name the project based on the model that will be tested e.g. "Model benchmark testing: WildObs National".
-  Set the Sequence cutoff to 0 seconds. This aims to prevent the software from creating sequences so that each image is assessed independently.
-  Define Tags in the project based on the scientific names of the species you are testing. Tags need to match with the species names used in the WIMP.
-  Configure the project to use the model you want to test
3) Create Deployments:
- You will need to create a Deployment for each of the species you are testing.
- Upload the relevant images into each deployment.
- Use the tags created earlier to assign to the deployment so you know which species it is supposed to be. This will be used by the script to match the species to the model predictions
- Repeat for each Project, uploading the same set of images to each
4) Run the uploaded images through the AI species recognition model
5) Once model processing is complete for all deployments, export the project data in Camtrap DP format
6) Download and extract (unzip) the exported data to a folder on your local computer
7) Use the folder path as input to this script

In [52]:
import pandas as pd
import os
import re

# -------- USER INPUT --------

#camtrap_folder = r"C:\Users\colin.broughton\Downloads\model-benchmarking-wildobs-national-20260318222241"
#camtrap_folder = r"C:\Users\colin.broughton\Downloads\model-benchmarking-awc-135-20260318222419"
camtrap_folder = r"C:\Users\colin.broughton\Downloads\model-benchmarking-speciesnet-v4-20260318221911"

output_report_folder = r"C:\Users\colin.broughton\Downloads\Model_Benchmarking_Report_Exports"
confidence_threshold = None  # e.g. 0.8 or None to disable
export_errors = True
data_source_location = "Bon Bon Reserve, SA"
data_collation_process = """
Collated 1000 images of each target species.
Images were manually validated as having at least one detection of the target species and no other species present in the image.
"""

# ----------------------------


# -----------------------------
# Load data
# -----------------------------
observations = pd.read_csv(os.path.join(camtrap_folder, "observations.csv"))
media = pd.read_csv(os.path.join(camtrap_folder, "media.csv"))
deployments = pd.read_csv(os.path.join(camtrap_folder, "deployments.csv"))


# -----------------------------
# Keep required columns
# -----------------------------
observations = observations[[
    "eventID",
    "deploymentID",
    "scientificName",
    "classificationProbability",
    "classifiedBy"
]].rename(columns={
    "classificationProbability": "confidence"
})

media = media[["mediaID", "mediaComments"]]

deployments = deployments[["deploymentID", "deploymentTags"]]

# Extract model name
model_name_series = observations["classifiedBy"].dropna().astype(str).str.strip()
model_name_series = model_name_series[model_name_series != ""]
model_name = model_name_series.iloc[0] if not model_name_series.empty else "Unknown Model"

# Extract species list
species_list = (
    deployments["deploymentTags"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.lower()
    .unique()
)

# Convert to sorted list
species_list = sorted(species_list)

def format_species_name(name):
    parts = str(name).split()
    if not parts:
        return name
    return " ".join([parts[0].capitalize()] + [p.lower() for p in parts[1:]])

species_list_display = [format_species_name(s) for s in species_list]

# -----------------------------
# Extract sequenceID
# -----------------------------
def extract_sequence(comment):
    if pd.isna(comment):
        return None
    match = re.search(r"sequenceID:([^\s]+)", str(comment))
    return match.group(1) if match else None


media["sequenceID"] = media["mediaComments"].apply(extract_sequence)


# -----------------------------
# Merge tables
# -----------------------------
merged = pd.merge(media, observations, left_on="sequenceID", right_on="eventID", how="left")
merged = pd.merge(merged, deployments, on="deploymentID", how="left")


# -----------------------------
# Normalize species names
# -----------------------------
merged["true_species"] = (
    merged["deploymentTags"]
    .astype(str)
    .str.strip()
    .str.lower()
)

merged["pred_species"] = (
    merged["scientificName"]
    .astype(str)
    .str.strip()
    .str.lower()
)


# -----------------------------
# Apply confidence threshold (optional)
# -----------------------------
if confidence_threshold is not None:
    merged = merged[merged["confidence"] >= confidence_threshold]


# -----------------------------
# Remove rows with missing truth
# -----------------------------
merged = merged[merged["true_species"].notna()]


# -----------------------------
# Evaluation metrics
# -----------------------------

"""
results = []

species_list = sorted(set(merged["true_species"]).union(set(merged["pred_species"])))

for species in species_list:

    tp = ((merged["true_species"] == species) & (merged["pred_species"] == species)).sum()
    fn = ((merged["true_species"] == species) & (merged["pred_species"] != species)).sum()
    fp = ((merged["true_species"] != species) & (merged["pred_species"] == species)).sum()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    f1 = (
        2 * (precision * recall) / (precision + recall)
        if (precision + recall) > 0 else 0
    )

    results.append({
        "species": species,
        "true_positives": int(tp),
        "false_negatives": int(fn),
        "false_positives": int(fp),
        "recall": round(recall, 4),
        "precision": round(precision, 4),
        "f1_score": round(f1, 4)
    })

"""

results = []

# Only include species that exist in ground truth
species_list = sorted(merged["true_species"].dropna().unique())

for species in species_list:

    tp = ((merged["true_species"] == species) & (merged["pred_species"] == species)).sum()
    fn = ((merged["true_species"] == species) & (merged["pred_species"] != species)).sum()
    fp = ((merged["true_species"] != species) & (merged["pred_species"] == species)).sum()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    f1 = (
        2 * (precision * recall) / (precision + recall)
        if (precision + recall) > 0 else 0
    )

    results.append({
        "species": species,
        "true_positives": int(tp),
        "false_negatives": int(fn),
        "false_positives": int(fp),
        "recall": round(recall, 4),
        "precision": round(precision, 4),
        "f1_score": round(f1, 4)
    })

results_df = pd.DataFrame(results).sort_values(by="f1_score", ascending=False)

def format_species_name(name):
    if pd.isna(name):
        return name
    
    parts = str(name).strip().split()
    
    if not parts:
        return name
    
    return " ".join([parts[0].capitalize()] + [p.lower() for p in parts[1:]])

results_df = pd.DataFrame(results).sort_values(by="f1_score", ascending=False)
results_df["species_display"] = results_df["species"].apply(format_species_name)

# Reorder columns (optional)
results_df = results_df[
    ["species_display", "true_positives", "false_negatives", "false_positives", "recall", "precision", "f1_score"]
].rename(columns={"species_display": "species"})


# -----------------------------
# Confusion matrix
# -----------------------------
conf_matrix = pd.crosstab(
    merged["true_species"],
    merged["pred_species"]
)

conf_matrix.index = conf_matrix.index.map(format_species_name)
conf_matrix.columns = conf_matrix.columns.map(format_species_name)


# -----------------------------
# Misclassified images export
# -----------------------------
if export_errors:
    errors = merged[merged["true_species"] != merged["pred_species"]]
    errors.to_csv("misclassified_images.csv", index=False)


# -----------------------------
# Output (Jupyter-friendly)
# -----------------------------
from IPython.display import display

print("\n=== MODEL EVALUATION RESULTS ===")
display(results_df)

print("\n=== CONFUSION MATRIX ===")
display(conf_matrix)

print("\nTotal observations analysed:", len(merged))


=== MODEL EVALUATION RESULTS ===


,species,true_positives,false_negatives,false_positives,recall,precision,f1_score
2,Vulpes vulpes,823,187,0,0.8149,1.0,0.8980
0,Felis catus,755,248,0,0.7527,1.0,0.8589
1,Oryctolagus cuniculus,28,988,0,0.0276,1.0,0.0536



=== CONFUSION MATRIX ===


pred_species,Bos taurus,Canidae,Canis familiaris,Canis latrans,Carnivora,Cervidae,Felidae,Felis,Felis catus,Lagorchestes conspicillatus,...,Odocoileus virginianus,Oryctolagus cuniculus,Procyon lotor,Sciuridae,Sylvilagus aquaticus,Sylvilagus audubonii,Sylvilagus brasiliensis,Sylvilagus floridanus,Urocyon cinereoargenteus,Vulpes vulpes
true_species,,,,,,,,,,,,,,,,,,,,,
Felis catus,2,2,2,2,31,1,57,13,755,0,...,0,0,1,1,0,0,0,0,4,0
Oryctolagus cuniculus,2,0,0,1,0,0,0,0,0,0,...,1,28,0,0,38,8,5,3,0,0
Vulpes vulpes,2,60,5,27,5,0,0,0,0,1,...,1,0,1,0,0,0,0,0,1,823



Total observations analysed: 3029


In [53]:
# generate a nicely formatted report that can be shared with others as a .html file

html_file = output_report_folder + "/" + f"Model_Benchmark_Test_Report_{model_name}.html"

style = """
<style>
body { font-family: Arial; margin: 40px; }
h1 { color: #2c3e50; }
h2 { color: #34495e; margin-top: 30px; }
h3 { color: #2c3e50; margin-top: 20px; }
p { max-width: 900px; line-height: 1.5; }
table { border-collapse: collapse; width: 80%; margin-top: 10px; }
th, td { border: 1px solid #ccc; padding: 8px; text-align: center; }
th { background-color: #f2f2f2; }
</style>
"""

# MathJax script
mathjax = """
<script src="https://cdn.jsdelivr.net/npm/mathjax@3/es5/tex-mml-chtml.js"></script>
"""

with open(html_file, "w", encoding="utf-8") as f:

    f.write("<html><head>")
    f.write(style)
    f.write(mathjax)
    f.write("</head><body>")

    f.write("<h1>Model Evaluation Report</h1>")
    
    f.write(f"<h2>Test Details</h2>")
    f.write(f"<p><b>Computer Vision (CV) model tested: </b>{model_name}<p>")
    f.write(f"<p><b>Source location of test images: </b>{data_source_location}</p>")
    #f.write(f"<p><b>Target species tested: </b>{species_list_display}</p>")
    f.write("<p><b>Species Evaluated:</b></p>")
    f.write("<ul>")
    for species in species_list_display:
        f.write(f"<li><i>{species}</i></li>")
    f.write("</ul>")
    f.write(f"<p><b>Data collation process used: </b>{data_collation_process}</p>")

    # -----------------------------
    # Explanation Section
    # -----------------------------

    f.write("<h2>Purpose and intended use of this report</h2>")
    f.write("""
        <ul>
            <li>Inform the potential suitability of a model for a given location and set of species relevant to the monitoring program.</li>
            <li>Inform the design of a validation workflow (manual human review) to ensure a sufficient level of accuracy can be reached to meet the goals of the monitoring program.</li>
            <li>Inform potential gaps in model performance and priorities for provision of additional training images to improve performance.</li>
        </ul>
    """)
    f.write("<h2>Disclaimer / limitations</h2>")
    f.write("""
        <ul>
            <li>The accuracy of the results presented in this report is primairly limited by the quality of the test data used.</li>
            <li>Please follow the recommended guidelines for collating a representative test dataset.</li>
            <li>The results presented in this report do not necessarily prove definitively that one model is better than another, but may provide guidence on selecting the most suitable model for your location and the species important to your monitoring program.</li>
        </ul>
    """)
    f.write("<p></p>")
    f.write("<h2>Understanding the Results</h2>")
    f.write("<p>Recall, precision, and F1 score are standard metrics used to evaluate the performance of machine learning models. Each metric has a precise definition and corresponding mathematical formula.</p>")

    f.write("<h3>Recall (Sensitivity)</h3>")
    f.write(r"<p>\[ \text{Recall} = \frac{\text{True Positives}}{\text{True Positives + False Negatives}} \]</p>")
    f.write("<p>Of all the images that actually contain the target species, how many did the model correctly identify?</p>")
    f.write("<p><b>Example: </b>If there are 100 images of Feral Cat within a dataset, and the model correctly detects 90 of them, recall = 0.9 (90%).</p>")
    f.write("<p><b>Interpretation:</b> Quantifying recall helps us to understand how many true detections of a given species might be missed by the model.</p>")

    f.write("<h3>Precision</h3>")
    f.write(r"<p>\[ \text{Precision} = \frac{\text{True Positives}}{\text{True Positives + False Positives}} \]</p>")
    f.write("<p>Of all the images the model predicts as the target species, how many are actually correct?</p>")
    f.write("<p><b>Example: </b>If the model makes 100 predictions of Feral Cat within a dataset, and only 90 of them are actually detections of Feral Cat, precision = 0.9 (90%).</p>")
    f.write("<p><b>Interpretation:</b> High precision means predictions are reliable. Low precision means more false detections.</p>")

    f.write("<h3>F1 Score</h3>")
    f.write(r"<p>\[ \text{F1 Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision + Recall}} \]</p>")
    f.write("<p>A balanced measure that combines both recall and precision.</p>")
    f.write("<p><b>Interpretation:</b> High F1 score indicates a good balance between detecting animals and avoiding false detections.</p>")
    f.write("<h3>Definition of Terms</h3>")

    f.write("""
    <table>
        <tr>
            <th>Term</th>
            <th>Definition</th>
            <th>Example</th>
        </tr>
        <tr>
            <td><b>True Positive (TP)</b></td>
            <td>The model correctly predicts the presence of a species.</td>
            <td>An image contains a fox, and the model predicts "fox".</td>
        </tr>
        <tr>
            <td><b>False Positive (FP)</b></td>
            <td>The model predicts a species, but that species is not actually present.</td>
            <td>An image contains a cat, but the model predicts "fox".</td>
        </tr>
        <tr>
            <td><b>False Negative (FN)</b></td>
            <td>The model fails to detect a species that is actually present.</td>
            <td>An image contains a fox, but the model predicts something else.</td>
        </tr>
    </table>
    """)
    
    f.write("""
    <p><b>Summary:</b></p>
    <ul>
        <li><b>True Positive</b> → Correct detection</li>
        <li><b>False Positive</b> → False alarm</li>
        <li><b>False Negative</b> → Missed detection</li>
    </ul>
    """)
    
    f.write("<h3>How to interpret results</h3>")
    f.write("""
    <table>
        <tr><th>Scenario</th><th>Interpretation</th></tr>
        <tr><td>High Recall, Low Precision</td><td>Finds most animals but includes many false positives</td></tr>
        <tr><td>Low Recall, High Precision</td><td>Accurate predictions but misses many animals</td></tr>
        <tr><td>High Recall, High Precision</td><td>Ideal performance</td></tr>
        <tr><td>Low Recall, Low Precision</td><td>Poor performance</td></tr>
    </table>
    """)
    f.write("""
    <h3>What is a Confusion Matrix?</h3>
    <ul>
        <li>A confusion matrix summarises the model’s performance by comparing actual vs predicted labels.</li>
        <li>In simple terms the confusion matrix shows what the model got right and wrong, broken down by type of error.</li>
        <li>Inspecting the confusion matrix helps us to understand the direction of error in cases where an image was incorrectly classified.</li>
        <li>Be aware that not all models define or name species the same way. In some cases a model may also generalise a species detections to a broader group e.g. Genus or Family. If recall is returning a very low number for a given species and model, inspection of confusion matrix may help to reveal inconsistencies in naming or categorisation.</li>
    </ul>
    """)

    # -----------------------------
    # Results Tables
    # -----------------------------
    f.write("<h2>Recall, Precision, and F1 Results</h2>")
    f.write(results_df.to_html(index=False))
    
    f.write("<h2>Confusion Matrix</h2>")
    f.write(conf_matrix.to_html())

    f.write("</body></html>")

print(f"Report saved to: {html_file}")

Report saved to: C:\Users\colin.broughton\Downloads\Model_Benchmarking_Report_Exports/Model_Benchmark_Test_Report_Speciesnet V4.0.html
